# Instrument Data Compilation
This notebook compiles raw instrument data into Parquet files.

This script only runs from within Meteowiss, because of the data store. It compiles raw data files stored on MeteoSwiss disk into parquet files. 
Incoming raw data files are first organized into folders, and a stastic is computed and displayed showing the number of recently incoming files.
Then, yearly files are generated for Meteo bulletins, Thermo zip files, NOAA CPD2 tarballs. Monthly files are generated for AE33 zip files, G2401 tarballs.

joerg.klausen@meteoswiss.ch

[TODO] Improve handling of erroneous files from g2401.compile_g2401_to_parquet

In [ ]:
from pathlib import Path
from processing.ae33 import AE33
from processing.fidas import Fidas
from processing.g2401 import G2401
from processing.meteo import Meteo
from processing.neph import Neph
from processing.thermo import Thermo
from processing.hmp110 import HMP110

from toolbox.utils import load_config

##############################
# Process MKN incoming files #
##############################
# read configuration
mkn = load_config(config_file="mch-mkn.yml")
target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

processors = {
    "tei49c": Thermo(name="tei49c"),
    "tei49i": Thermo(name="tei49i"),
    "g2401": G2401(),
    "ne300": Neph(name="ne300"),
    "ae33/data": AE33(),
    "hmp110-inlet": HMP110(name="hmp110-inlet"),
    "hmp110-ae33": HMP110(name="hmp110-ae33"),
    "fidas": Fidas(),
    "meteo": Meteo(name="vrxa00"),
}

for name, processor in processors.items():
    print(f"▶ Processing {name.upper()} ...", flush=True)
    processor.compile_to_parquet(
        source=Path(mkn['root']) / mkn['branches']['incoming'] / name,
        target=target / "level1" / "mkn",
        archive=Path(mkn['root']) / mkn['branches']['archive'] / name,
        issues=Path(mkn['root']) / mkn['branches']['issues'] / name,
        split=mkn[name]['split_parquet']
    )

In [ ]:
from pathlib import Path
from processing.ae31 import AE31
from processing.fidas import Fidas
from processing.meteo import Meteo
from processing.neph import Neph
from processing.thermo import Thermo
from processing.hmp110 import HMP110
from processing.avo import AVO

from toolbox.utils import load_config

##############################
# Process NRB incoming files #
##############################
# read configuration
nrb = load_config(config_file="mch-nrb.yml")
log_file = str(Path("logs") / "nrb.log")
target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

processors = {
    "49i": Thermo(name="49i", log_file=log_file),
    "fidas": Fidas(log_file=log_file),
    "aurora3000": Neph(name="aurora3000", log_file=log_file),
    "ae31": AE31(log_file=log_file),
    "avo": AVO(log_file=log_file),    
    "hmp110-ae31": HMP110(name="hmp110-ae31", log_file=log_file),
    "hmp110-inlet": HMP110(name="hmp110-inlet", log_file=log_file),
    "hmp110-lab": HMP110(name="hmp110-lab", log_file=log_file),
    # "meteo": Meteo(name="vrxa00"),
}

for name, processor in processors.items():
    print(f"▶ Processing {name.upper()} ...")
    processor.compile_to_parquet(
        source=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['incoming'] / name,
        target=target / "level1" / "nrb",
        archive=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['archive'] / name,
        issues=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['issues'] / name,
        split=nrb['nrb-aq'][name]['split_parquet']
    )    

In [ ]:
from pathlib import Path
from processing.thermo import Thermo

from toolbox.utils import load_config

##############################
# Process BUC incoming files #
##############################
# read configuration
buc = load_config(config_file="mch-buc.yml")
target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

processors = {
    "49i": Thermo(name="49i"),
}

for name, processor in processors.items():
    print(f"▶ Processing {name.upper()} ...", flush=True)
    processor.compile_to_parquet(
        source=Path(buc['root']) / buc['branches']['incoming'] / name,
        target=target / "level1" / "buc",
        archive=Path(buc['root']) / buc['branches']['archive'] / name,
        issues=Path(buc['root']) / buc['branches']['issues'] / name,
        split=buc[name]['split_parquet']
    )